# Load simulated data

In [ ]:
import datetime

import numpy as np
from stonesoup.types.state import GaussianState

from theia.coordinates import CoordinateTransformations
from theia.simulation.logging import LogLoader

# How to go from the 6D state space (positions + velocities) to the
# 3D measurement space (positions only).
# measurement_model = LinearGaussian(
#     ndim_state=6,
#     mapping=(0, 2, 4),
#     noise_covar=np.identity(3),
# )

loader = LogLoader("log_opensky.json")

# Single Target

## Select a single radar and a single target

In [ ]:
RADAR_ID = 0
TARGET_ID = 66
TARGET_ID2 = 3

detections = [
    d
    for d in loader.blue_monostatic_radar_detections
    if d.metadata["radar_id"] == RADAR_ID and d.metadata["target_id"] in [TARGET_ID, -2]
]
detections2 = [
    d
    for d in loader.blue_monostatic_radar_detections
    if d.metadata["radar_id"] == RADAR_ID and d.metadata["target_id"] == TARGET_ID2
]

radar = next(
    radar for radar in loader.blue_monostatic_radars if radar.receiver.id == RADAR_ID
)
ground_truth = loader.red_target_ground_truth[TARGET_ID]
ground_truth2 = loader.red_target_ground_truth[TARGET_ID2]

## Build tracker

In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.predictor.kalman import ExtendedKalmanPredictor
from stonesoup.updater.kalman import ExtendedKalmanUpdater

q = 1

transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(q), ConstantVelocity(q), ConstantVelocity(q)]
)

predictor = ExtendedKalmanPredictor(transition_model)
updater = ExtendedKalmanUpdater()

# Prior: Zero velocity, position of first detection.
first_detection = sorted(
    [d for d in detections if d.metadata["target_id"] in [TARGET_ID]],
    key=lambda d: d.timestamp,
)[0]

p = first_detection.measurement_model.inverse_function(first_detection)

prior = GaussianState(
    [
        [p[0, 0]],
        [0],
        [p[2, 0]],
        [0],
        [p[4, 0]],
        [0],
    ],
    covar=np.identity(6),
    timestamp=first_detection.timestamp,
)

## Track

In [ ]:
times = [state.timestamp for state in ground_truth]

### Working: Single target tracking

In [ ]:
from stonesoup.types.track import Track
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.dataassociator.neighbour import NearestNeighbour
from stonesoup.types.detection import Clutter

mahalanobis_miss_distance = 50

hypothesiser = DistanceHypothesiser(
    predictor,
    updater,
    measure=Mahalanobis(),
    missed_distance=mahalanobis_miss_distance,
)
data_associator = NearestNeighbour(hypothesiser)

post = prior
ground_truth = Track([prior])
for t in times:
    # prediction = predictor.predict(prior, timestamp=detection.timestamp)
    # hypothesis = SingleHypothesis(prediction, detection)

    detections_at_t = [d for d in detections if d.timestamp == t]
    hypotheses = data_associator.associate([ground_truth], detections_at_t, t)
    hypothesis = hypotheses[ground_truth]

    if hypothesis.measurement:
        before = post
        post = updater.update(hypothesis)
        ground_truth.append(post)
        # dist = Mahalanobis()(post, before)
        # print(f"Mahalanobis distance true->true: {dist:.3f}")
    else:
        # When data associator says no detections are good enough,
        # we'll keep the prediction.
        ground_truth.append(hypothesis.prediction)

    # clutter_at_t = [d for d in detections_at_t if type(d) is Clutter]
    # clutter_at_t_considered = [
    #     c for c in clutter_at_t if Mahalanobis()(post, c) < mahalanobis_miss_distance
    # ]
    # print(
    #     f"Number of clutter plots considered as detections: {len(clutter_at_t_considered):3d} / {len(clutter_at_t):3d} ({len(clutter_at_t_considered) / len(clutter_at_t) * 100:.1f}%)"
    # )

In [ ]:
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(times[::2])
# Order: [x, vx, y, vy, z, vz] -> Need indices 0, 2, 4
plotter.plot_ground_truths((ground_truth), (0, 2))
plotter.plot_measurements(detections, (0, 2), show_clutter=True)
plotter.plot_tracks(ground_truth, (0, 2), uncertainty=True)
plotter.fig

In [ ]:
import pandas as pd


pd.DataFrame(
    [
        {
            "elevation": d.state_vector[0],
            "azimuth": d.state_vector[1],
            "range": d.state_vector[2],
            "sigma_elevation": np.sqrt(d.measurement_model.noise_covar[0, 0]),
            "sigma_azimuth": np.sqrt(d.measurement_model.noise_covar[1, 1]),
            "sigma_range": np.sqrt(d.measurement_model.noise_covar[2, 2]),
            "snr_dB": d.metadata["snr"],
        }
        for d in detections
    ]
)

# Multiple targets (in progress)

## Prepare data

In [ ]:
RADAR_ID = 0
TARGET_ID = 66
TARGET_ID2 = 3

detections = [
    d
    for d in loader.blue_monostatic_radar_detections
    if d.metadata["radar_id"] == RADAR_ID
    and d.metadata["target_id"] in [TARGET_ID, TARGET_ID2, -2]
]

radar = next(
    radar for radar in loader.blue_monostatic_radars if radar.receiver.id == RADAR_ID
)   
ground_truths = [
    loader.red_target_ground_truth[TARGET_ID],
    loader.red_target_ground_truth[TARGET_ID2],
]

times = []
for gt in ground_truths:
    times.extend([state.timestamp for state in gt.states])
times = sorted(list(set(times)))

## Track using theia wrapper

In [ ]:
from theia.stonesoup_interface import MonostaticDetectionFactory


temporal_detections = {}
for detection in detections:
    group = temporal_detections.get(detection.timestamp, list())
    group.append(detection)
    temporal_detections[detection.timestamp] = group

for time, group in temporal_detections.items():
    temporal_detections[time] = set(group)

In [ ]:
from theia.simulation.tracking import MonostaticSingleSensorTracker


tracker = MonostaticSingleSensorTracker()

In [ ]:
N_TIMES = 170

In [ ]:
from tqdm import tqdm


for time in tqdm(times[:N_TIMES]):
    detections_at_time = temporal_detections.get(time, [])
    tracker.add_detections(detections_at_time)

## Plot theia tracking

In [ ]:
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(times[:N_TIMES:2])
# Order: [x, vx, y, vy, z, vz] -> Need indices 0, 2, 4
plotter.plot_ground_truths((ground_truths), (0, 2))
plotter.plot_measurements(detections, (0, 2), show_clutter=True)
plotter.plot_tracks(
    # [track for track in all_tracks if len(track) > 3], (0, 2), uncertainty=True
    tracker._tracks,
    (0, 2),
    # tracker._tracks, (0, 2),
)
plotter.fig

## Track with raw stonesoup

In [ ]:
from stonesoup.deleter.error import CovarianceBasedDeleter
from stonesoup.deleter.time import UpdateTimeStepsDeleter
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.dataassociator.neighbour import NearestNeighbour
from stonesoup.types.detection import Clutter


a_max = 30  # m/s², maximum expected acceleration (e.g. 3g for a fighter jet)
# q = a_max**2  # variance of acceleration per second
q = 1
mahalanobis_miss_distance = 50
deletion_covariance_threshold = 200_000

transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(q), ConstantVelocity(q), ConstantVelocity(q)]
)
predictor = ExtendedKalmanPredictor(transition_model)
updater = ExtendedKalmanUpdater()

hypothesiser = DistanceHypothesiser(
    predictor,
    updater,
    measure=Mahalanobis(),
    missed_distance=mahalanobis_miss_distance,
)
data_associator = NearestNeighbour(hypothesiser)

# Delete tracks once their covariance trace exceeds 4.
deleter = CovarianceBasedDeleter(covar_trace_thresh=deletion_covariance_threshold)
# track_deleter = UpdateTimeStepsDeleter(time_steps_since_update=3)  # confirmed tracks
init_deleter = UpdateTimeStepsDeleter(
    time_steps_since_update=1
)  # tentative tracks — kill immediately if no match on next scan

# Generate new track from unassigned detections
# if at least two detections are assigned to it.
v_max = 300  # m/s, maximum expected target speed
sigma_v = v_max / 3  # 3-sigma covers the full speed range

initiator = MultiMeasurementInitiator(
    # Used by default, but measured components are replaced by the detection
    # that initialises the track.
    prior_state=GaussianState(
        [[0], [0], [0], [0], [0], [0]],
        np.diag([0.0, sigma_v**2, 0.0, sigma_v**2, 0.0, sigma_v**2]),
    ),
    deleter=init_deleter,
    # deleter=deleter,
    data_associator=data_associator,
    updater=updater,
    min_points=5,
)

In [ ]:
# Adapted from stonesoup tutorial:
# https://stonesoup.readthedocs.io/en/v1.8/auto_tutorials/09_Initiators_%26_Deleters.html
from tqdm import tqdm


tracks = set()
all_tracks = set()

for time in tqdm(times[:N_TIMES]):
    detections_at_time = set([d for d in detections if d.timestamp == time])
    if (len(detections_at_time)) == 0:
        continue
    n_true = len([d for d in detections_at_time if type(d) != Clutter])
    # print("------------------------")
    # print(
    #     f"\n=== {time} | true detections: {n_true}, detections: {len(detections_at_time)}, active tracks: {len(tracks)} ==="
    # )
    hypotheses = data_associator.associate(tracks, detections_at_time, time)
    associated_measurements = set()
    for track in tracks:
        hypothesis = hypotheses[track]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            track.append(post)
            associated_measurements.add(hypothesis.measurement)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            track.append(hypothesis.prediction)

        # trace = np.trace(track.covar)
        # print(f"  track covariance trace: {trace:.1f}")

    # Carry out deletion and initiation
    before_delete = len(tracks)
    # tracks -= track_deleter.delete_tracks(tracks)
    tracks -= deleter.delete_tracks(tracks)
    # print(f"  deleted {before_delete - len(tracks)} tracks")

    unassociated = detections_at_time - associated_measurements
    # print(f"  unassociated detections passed to initiator: {len(unassociated)}")

    if unassociated:
        new_tracks = initiator.initiate(unassociated, time)
        # print(f"  new tracks initiated: {len(new_tracks)}")

        tracks |= new_tracks
        all_tracks |= tracks

In [ ]:
len(all_tracks)
# len(tracker.get_tracks())

In [ ]:
len(tracks)

## Plot raw tracking

In [ ]:
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(times[::2])
# Order: [x, vx, y, vy, z, vz] -> Need indices 0, 2, 4
plotter.plot_ground_truths((ground_truths), (0, 2))
plotter.plot_measurements(detections, (0, 2), show_clutter=True)
plotter.plot_tracks(
    # [track for track in all_tracks if len(track) > 3], (0, 2), uncertainty=True
    tracks,
    (0, 2),
    # tracker._tracks, (0, 2),
)
plotter.fig

## Plot matplotlib

In [ ]:
from theia.measurement import MonostaticMeasurementTransformations
from stonesoup.types.detection import Detection

def stonesoup_detections_to_xyz(detections: set[Detection]) -> np.ndarray:
    detections_xyz = np.empty((len(detections), 3), dtype=np.float32)
    for i, d in enumerate(detections):
        elevation, azimuth, range_m = d.state_vector.flatten().tolist()
        detections_xyz[i, :] = (
            MonostaticMeasurementTransformations.elevation_azimuth_range_to_cartesian(
                d.measurement_model.p_radar,
                elevation,
                azimuth,
                range_m,
            )
        )
    return detections_xyz

In [ ]:
from theia.types import ConstantRcsModel, Trajectory

def stonesoup_track_to_trajectory(track):
    times = []
    lats = []
    lons = []
    alts = []
    vxs = []
    vys = []
    vzs = []
    for state in track.states:
        times.append(state.timestamp)
        lat, lon, alt = CoordinateTransformations.cartesian_to_geodetic(
            state.state_vector[0],
            state.state_vector[2],
            state.state_vector[4],
        )
        lats.append(lat)
        lons.append(lon)
        alts.append(alt)
        vx, vy, vz = state.state_vector[[1, 3, 5]]
        vxs.append(vx)
        vys.append(vy)
        vzs.append(vz)
        # ground_truth_trajectory.append()

    return Trajectory(
        target_id=TARGET_ID,
        times=times,
        lats=lats,
        lons=lons,
        alts=alts,
        vxs=vxs,
        vys=vys,
        vzs=vzs,
        cross_section_model=ConstantRcsModel(rcs=np.nan),
    )


trajectory = stonesoup_track_to_trajectory(ground_truths[0])
tracked_trajectory = tracker.get_tracks()[0]

In [ ]:
time

In [ ]:
times[0]

In [ ]:
ground_truth.states[0].timestamp

In [ ]:
datetime.datetime.fromtimestamp(tracked_trajectory._times[0], tz=datetime.timezone.utc)

In [ ]:
from matplotlib import animation
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))

time = times[0]

detections = temporal_detections.get(time, set())
detections_xyz = stonesoup_detections_to_xyz(detections)
gt = trajectory(time)
track_state = tracked_trajectory(time)
gt_xyz = CoordinateTransformations.geodetic_to_cartesian(gt.lat, gt.lon, gt.alt)

return

(detections_artist,) = ax.plot(
    detections_xyz[:, 0],
    detections_xyz[:, 1],
    "o",
    markersize=1,
)
(ground_truth_artist,) = ax.plot(gt_xyz[0], gt_xyz[1], "*", markersize=7, label="Ground truth")
(track_artist,) = ax.plot(track_state[0], track_state[2], "x", markersize=7, label="Track")

ax.legend()

def update(frame: int):
    time = times[frame]
    detections = temporal_detections.get(time, set())
    detections_xyz = stonesoup_detections_to_xyz(detections)
    gt = trajectory(time)
    track_state = ground_truth(time)
    gt_xyz = CoordinateTransformations.geodetic_to_cartesian(gt.lat, gt.lon, gt.alt)

    detections_artist.set_xdata(detections_xyz[:, 0])
    detections_artist.set_ydata(detections_xyz[:, 1])

    ground_truth_artist.set_xdata([gt_xyz[0]])
    ground_truth_artist.set_ydata([gt_xyz[1]])

    track_artist.set_xdata([track_state[0]])
    track_artist.set_ydata([track_state[2]])


ani = animation.FuncAnimation(fig=fig, func=update, frames=40, interval=30)
from IPython.display import HTML
HTML(ani.to_jshtml())

In [ ]:
gt

In [ ]:
detections_artist.set

## Plot on map

In [ ]:
from typing import Iterable
import shapely
from stonesoup.types.state import State


def stonesoup_states_to_shapely(states: Iterable[State]) -> shapely.LineString:
    coordinates = []
    for state in states:
        lat, lon, alt = CoordinateTransformations.cartesian_to_geodetic(
            state.state_vector[0],
            state.state_vector[2],
            state.state_vector[4],
        )
        coordinates.append((lon, lat, alt))
    return shapely.LineString(coordinates)

In [ ]:
true_trajectory_xyz = np.stack(
    [state.state_vector.flatten() for state in ground_truth.states]
)
true_trajectory_times = [state.timestamp for state in ground_truth.states]
tracked_trajectory_xyz = np.stack(
    [state.state_vector.flatten() for state in ground_truth.states]
)
tracked_trajectory_times = [state.timestamp for state in ground_truth.states]

# diff = true_trajectory - tracked_trajectory

In [ ]:
from matplotlib import pyplot as plt


fig, axes = plt.subplots(ncols=3)

t0 = true_trajectory_times[0]

for i in range(3):
    ax = axes[i]
    ax.plot(true_trajectory_times, true_trajectory_xyz[:, i * 2])
    ax.plot(tracked_trajectory_times, tracked_trajectory_xyz[:, i * 2], "--")

    for detection in detections:
        ax.axvline(detection.timestamp, color="red", linewidth=0.5)
    ax.tick_params(axis="x", rotation=90)
    ax.set_xlim(
        [t0 + datetime.timedelta(minutes=5), t0 + datetime.timedelta(minutes=13)]
    )

fig.tight_layout()

In [ ]:
from theia.distance import get_2d_distance_between_locs_heights

ground_truth_lonlatalt = stonesoup_states_to_shapely(ground_truth.states).coords

latlon_of_interest = (47.4872, 8.9635)
diffs = [
    get_2d_distance_between_locs_heights(
        latlon_of_interest[0],
        latlon_of_interest[1],
        0,
        c[1],
        c[0],
        0,
    )
    for c in ground_truth_lonlatalt
]

i = int(np.argmin(diffs))
alt = ground_truth_lonlatalt[i][2]
alt

In [ ]:
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range

RCS = 2.0

coverage = calculate_coverage(
    radar.receiver.point,
    calculate_maximum_monostatic_range(radar, RCS),
    alt,
)

In [ ]:
from theia.mapping import RadarMap

mapper = RadarMap(
    radars={"radar": radar},
    # paths={
    #     "ground truth": stonesoup_states_to_shapely(ground_truth.states),
    #     # "tracked": stonesoup_states_to_shapely(track.states),
    # },
    paths={
        i: stonesoup_states_to_shapely(gt)
        for i, gt in loader.red_target_ground_truth.items()
    },
    polygons={"coverage": coverage},
)
mapper.to_plotly_map()

# 2 targets, initiate / delete tracks

## Select single radar, multiple targets

In [ ]:
import pandas as pd
from stonesoup.types.detection import Detection


RADAR_ID = 0
TARGET_ID = 51
TARGET_ID2 = 66

detections_filtered = [
    d
    for d in loader.blue_monostatic_radar_detections
    if (d.metadata["radar_id"] == RADAR_ID)
    and (d.metadata["target_id"] in [TARGET_ID, TARGET_ID2])
]

detections_per_time: dict[pd.Timestamp, list[Detection]] = {}
for detection in detections_filtered:
    detections = detections_per_time.get(detection.timestamp)
    if detections is None:
        detections_per_time[detection.timestamp] = [detection]
    else:
        detections_per_time[detection.timestamp].append(detection)


radar = next(
    radar for radar in loader.blue_monostatic_radars if radar.receiver.id == RADAR_ID
)
ground_truth = loader.red_target_ground_truth[TARGET_ID]
ground_truth2 = loader.red_target_ground_truth[TARGET_ID2]

In [ ]:
# mapper = RadarMap(
#     radars={"radar": radar},
#     paths={
#         i: stonesoup_states_to_shapely(ground_truth.states)
#         for i, ground_truth in loader.red_target_ground_truth.items()
#     },
#     polygons={"coverage": coverage},
# )
# mapper.to_map()

## Build tracker

In [ ]:
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
from stonesoup.deleter.error import CovarianceBasedDeleter
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.measures import Mahalanobis
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

# Uncertainty for the transition.
q = 0.05>

transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(q), ConstantVelocity(q), ConstantVelocity(q)]
)

predictor = KalmanPredictor(transition_model)
updater = KalmanUpdater(measurement_model)

# Generate hypotheses: Which target might belong to which track?
# Exclude hypotheses with Mahalanobis distance >= 3.
hypothesiser = DistanceHypothesiser(
    predictor,
    updater,
    measure=Mahalanobis(),
    missed_distance=3,
)
# Select the hypotheses (one detection per track) based on Mahalanobis proximity.
data_associator = GlobalNearestNeighbour(hypothesiser)

# Delete tracks once their covariance trace exceeds 4.
deleter = CovarianceBasedDeleter(covar_trace_thresh=4)

# Generate new track from unassigned detections
# if at least two detections are assigned to it.
initiator = MultiMeasurementInitiator(
    # Used by default, but measured components are replaced by the detection
    # that initialises the track.
    prior_state=GaussianState(
        [[0], [0], [0], [0], [0], [0]],
        np.diag([0, 1, 0, 1, 0, 1]),
    ),
    measurement_model=measurement_model,
    deleter=deleter,
    data_associator=data_associator,
    updater=updater,
    min_points=2,
)


## Track

In [ ]:
times = [snapshot.time for snapshot in loader._snapshots]

In [ ]:
# Based on stonesoup tutorial 9 - "Initiators & Deleters" (retrieved 2026-02-20)
# https://stonesoup.readthedocs.io/en/v1.8/auto_tutorials/09_Initiators_&_Deleters.html
tracks = set()
all_tracks = set()

for time in times:
    detections = detections_per_time.get(time, [])
    # if len(detections) > 1:
    #     print(time, len(detections))
    # Calculate all hypothesis pairs and associate the elements in the best subset to the tracks.
    hypotheses = data_associator.associate(
        tracks,
        detections,
        time,
    )
    associated_measurements = set()
    for ground_truth in tracks:
        hypothesis = hypotheses[ground_truth]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            ground_truth.append(post)
            associated_measurements.add(hypothesis.measurement)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            ground_truth.append(hypothesis.prediction)

    # Carry out deletion and initiation
    tracks -= deleter.delete_tracks(tracks)
    tracks |= initiator.initiate(
        set(detections) - associated_measurements,
        time,
    )
    all_tracks |= tracks

In [ ]:
plotter = AnimatedPlotterly(times[::5], tail_length=0.3)
plotter.plot_ground_truths([ground_truth, ground_truth2], [0, 2])
plotter.plot_measurements(detections_filtered, [0, 2])
plotter.plot_tracks(all_tracks, [0, 2], uncertainty=True)
plotter.fig

In [ ]:
from theia.types import ConstantRcsModel, Trajectory
from stonesoup.types.groundtruth import GroundTruthPath


def trajectory_from_stonesoup(
    ground_truth_path: GroundTruthPath,
    cross_section_model: ConstantRcsModel,
) -> Trajectory:
    points = [
        CoordinateTransformations.cartesian_to_geodetic(
            s.state_vector[0],
            s.state_vector[2],
            s.state_vector[4],
        )
        for s in ground_truth_path.states
    ]
    return Trajectory(
        target_id=ground_truth_path.states[0].metadata["target_id"],
        times=[s.timestamp.to_pydatetime() for s in ground_truth_path.states],
        lats=[p[0] for p in points],
        lons=[p[1] for p in points],
        alts=[p[2] for p in points],
        vxs=[s.state_vector[1] for s in ground_truth_path.states],
        vys=[s.state_vector[3] for s in ground_truth_path.states],
        vzs=[s.state_vector[5] for s in ground_truth_path.states],
        cross_section_model=cross_section_model,
    )

In [ ]:
trajectories = [
    trajectory_from_stonesoup(ground_truth, ConstantRcsModel(rcs=1.0))
    for ground_truth in loader.red_target_ground_truth.values()
]

In [ ]:
from theia.mapping import plot_trajectories


fig = plot_trajectories(
    times[::5],
    {"Rx": radar},
    [t for t in trajectories if t.target_id in [TARGET_ID, TARGET_ID2]],
)

fig.write_html("opensky.html")

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range

In [ ]:
from theia.types import Point, Polarization, Radar, Receiver, Transmitter


calculate_maximum_monostatic_range(
    Radar(
        transmitter=Transmitter(
            id=0,
            point=Point(lat=0, lon=0, alt=0),
            power=20_000,
            erp=20_000,
            antenna_height=0.0,
            antenna_diameter=2.0,
            frequency=1000,
            pulse_width=1.0,
            polarization=Polarization.VERTICAL,
            bandwidth=100.0,
        ),
        receiver=Receiver(
            id=0,
            point=Point(lat=0, lon=0, alt=0),
            antenna_height=0.0,
            diameter=2.0,
            cpi_pulses=1,
            pfa=1e-6,
            min_elevation=-20,
            max_elevation=60,
            rotation_time=1.0,
            bandwidth=100.0,
        ),
    ),
    1.0,
)

In [ ]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Draw sphere.
theta = np.linspace(0, 2 * np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)
r = 1

x = r * np.sin(phi) * np.cos(theta)
y = r * np.sin(phi) * np.sin(theta)
z = r * np.cos(phi)

fig.add_trace(
    go.Surface(
        x=x,
        y=y,
        z=z,
        opacity=0.3,
        colorscale="Blues",
        showscale=False,
    )
)

# Axis arrows.
L = 1.5

axes = [
    ([0, L], [0, 0], [0, 0], "red"),  # X
    ([0, 0], [0, L], [0, 0], "green"),  # Y
    ([0, 0], [0, 0], [0, L], "blue"),  # Z
]

for x_line, y_line, z_line, color in axes:
    fig.add_trace(
        go.Scatter3d(
            x=x_line,
            y=y_line,
            z=z_line,
            mode="lines",
            line=dict(color=color, width=6),
            showlegend=False,
        )
    )

# Arrowheads using cones
fig.add_trace(
    go.Cone(
        x=[L, 0, 0],
        y=[0, L, 0],
        z=[0, 0, L],
        u=[0.001, 0, 0],
        v=[0, 0.001, 0],
        w=[0, 0, 0.001],
        sizemode="absolute",
        sizeref=0.15,
        showscale=False,
        # colorscale=[[0, 'red'], [1, 'red']]
    )
)

# Add a point.
theta0 = np.pi / 4  # 45° azimuth
phi0 = np.pi / 3  # 60° polar angle
r0 = 1  # sphere radius

# ---- Convert to Cartesian ----
x0 = r0 * np.sin(phi0) * np.cos(theta0)
y0 = r0 * np.sin(phi0) * np.sin(theta0)
z0 = r0 * np.cos(phi0)

# ---- Radial line ----
fig.add_trace(
    go.Scatter3d(
        x=[0, x0],
        y=[0, y0],
        z=[0, z0],
        mode="lines",
        line=dict(color="black", width=8),
        showlegend=False,
    )
)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-1.7, 1.7]),
        yaxis=dict(range=[-1.7, 1.7]),
        zaxis=dict(range=[-1.7, 1.7]),
        aspectmode="cube",
    ),
    title="Transparent Sphere with Axis Arrows",
)